# E2.6 · Incident and disclosure obligations

**Function E — Governance, Risk, Compliance & the CISO Office → The Regulatory & Compliance Lead**  ·  *Security of AI*

---

**Risk.** Materiality assessed for an autonomous actor with a human-actor playbook.

**Control.** Coordinate with D2 in hour one.

**This lab.** Draft the notification for an autonomous actor.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("E2.6"))

Incident and disclosure obligations, applied to an incident whose actor was an agent. The clock and the attribution interact badly.

In [ ]:
from cybercommons import ir
import time

t0 = time.time()
tl = ir.Timeline()
tl.add(t0,      "alice", "alice",       "login")
tl.add(t0 + 40, "alice", "patch-agent", "read_file", "/work/customer_export.csv")
tl.add(t0 + 41, "alice", "patch-agent", "http_get",  "https://collect.example.com/")
r = ir.reconstruct(tl)
print("attribution:", r["attribution"])
print("hidden actors:", r["hidden_actors"])

Awareness starts when you know a reportable event *may* have occurred — not when you have finished attributing it. Broken attribution therefore consumes the clock rather than pausing it.

In [ ]:
SCENARIOS = {
 "attribution sound, report at 20h":   (t0 + 4 * 3600,  t0 + 20 * 3600),
 "attribution broken, 3 days to scope": (t0 + 70 * 3600, t0 + 76 * 3600),
}
for name, (contained, reported) in SCENARIOS.items():
    c = ir.clock(t0, contained, reported, deadline_hours=72)
    print(f"{name:38s} report {c['hours_to_report']:>5.1f}h met={c['met']} "
          f"margin {c['margin_hours']:+.1f}h")

### Expect

Attribution is reported BROKEN with `patch-agent` hidden. The sound scenario meets the 72-hour deadline with a wide margin; the broken one misses it.

### Your turn

Draft the disclosure sentence you would send when you know an agent acted but cannot yet say which one. Write it now, not during the incident.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/E2.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*